# Prompting Tips
### 5 practical techniques before you dive into few-shot and chain-of-thought

| Tip | What it teaches |
|---|---|
| 1 | Be specific — add details to your prompt |
| 2 | Use the system prompt for persistent instructions |
| 3 | Give the model a role (persona prompting) |
| 4 | Use delimiters to separate input from instructions |
| 5 | Specify your output format and length |


In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
from openai import OpenAI

# OpenRouter uses the OpenAI-compatible API
# Set OPENROUTER_API_KEY in your environment variables
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Model to use throughout this course
# OpenRouter format: "provider/model-name"
MODEL = "anthropic/claude-sonnet-4-5"


In [ ]:
def get_completion(prompt, system="You are a helpful assistant.", temperature=0):
    """
    Simple wrapper around OpenRouter API.

    Args:
        prompt      : the user message
        system      : the system prompt
        temperature : 0 = deterministic, 1 = creative
    
    Returns:
        The model's response as a plain string.
    """
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ]
    )
    return response.choices[0].message.content


---
## Tip 1 — Be Specific

**Vague prompts give vague answers.**  
Adding a small constraint (format, length, audience) makes the output much more useful.

In [ ]:
# ❌ Vague — the model decides format and length
vague = get_completion("What is the capital of France?")
print("Vague prompt:")
print(vague)


In [ ]:
# ✅ Specific — tell it exactly what you want
specific = get_completion("What is the capital of France? Just provide the answer, nothing else.")
print("Specific prompt:")
print(specific)


**The rule:** If you can imagine 5 different valid answers, the prompt is too vague.  
Add constraints: output format, audience level, or length.
```python
"Respond in JSON"         # format
"Explain to a beginner"   # audience  
"In one sentence"         # length
```


---
## Tip 2 — Use the System Prompt for Persistent Instructions

The system prompt sets the model's behavior for the entire conversation — you write it once.

In [ ]:
# Without system prompt — model decides tone and format
without = get_completion("What is backpropagation?")
print("Without system prompt:")
print(without[:300], "...")


In [ ]:
# With system prompt — behavior is controlled
with_system = get_completion(
    prompt="What is backpropagation?",
    system=(
        "You are a professor teaching 2nd-year computer engineering students. "
        "Explain concepts in 2-3 sentences using simple language. No formulas."
    )
)
print("With system prompt:")
print(with_system)


---
## Tip 3 — Give the Model a Role

Assigning a persona steers the model's tone, vocabulary, and detail level.

In [ ]:
question = "What is overfitting in machine learning?"

roles = [
    "You are a university professor. Give a formal, precise definition.",
    "You are explaining to a high school student with no ML background. Use a simple analogy.",
    "You are a senior ML engineer. Be concise and practical.",
]

for role in roles:
    answer = get_completion(question, system=role)
    print(f"ROLE: {role[:60]}...")
    print(f"ANSWER: {answer}")
    print("-" * 70)


**Useful roles for NLP projects:**
```python
"You are an NLP expert specialising in biomedical text."
"You are a strict JSON formatter. Output only valid JSON, nothing else."
"You are a customer support agent. Reply in a friendly, professional tone."
```


---
## Tip 4 — Use Delimiters to Separate Input from Instructions

When your prompt contains both **instructions** and **data to process**, delimiters prevent confusion.

In [ ]:
# ❌ Without delimiters — model might mix instruction and data
no_delim = get_completion(
    "Summarise this in one sentence. "
    "Large language models have achieved remarkable performance. "
    "They are trained on massive datasets. However they can hallucinate."
)
print("Without delimiters:")
print(no_delim)


In [ ]:
# ✅ With XML delimiters — clear separation
with_delim = get_completion(
    "Summarise the text below in one sentence.\n\n"
    "<text>\n"
    "Large language models have achieved remarkable performance. "
    "They are trained on massive datasets. However they can hallucinate.\n"
    "</text>"
)
print("With XML delimiters:")
print(with_delim)


In [ ]:
# Multiple inputs using XML tags
prompt = (
    "Classify the sentiment of each review as positive, negative, or neutral.\n\n"
    "<review_1>\nThe product works great, very happy!\n</review_1>\n\n"
    "<review_2>\nCompletely broken on arrival.\n</review_2>\n\n"
    "<review_3>\nIt arrived. Does what it says.\n</review_3>"
)
print(get_completion(prompt))


---
## Tip 5 — Specify Output Format and Length

For NLP pipelines you usually need **machine-readable, predictable output**.

In [ ]:
email = """
Hi, I have been a customer for 3 years and my last two orders arrived late.
The tracking page says delivered but I never received them. Very frustrated.
"""

# ❌ No format specified
free = get_completion(
    f"Respond to this customer email:\n{email}",
    system="You are a customer support agent."
)
print("Free form:")
print(free)


In [ ]:
# ✅ Structured format — usable in a pipeline
structured = get_completion(
    f"Respond to this customer email:\n{email}",
    system=(
        "You are a customer support agent. "
        "Your response must follow this exact format:\n"
        "Intent: <shipping_issue | billing | product_defect | general_inquiry>\n"
        "Priority: <high | medium | low>\n"
        "Reply: <your 1-2 sentence reply to the customer>"
    )
)
print("Structured:")
print(structured)


In [ ]:
# Parse the structured output programmatically
result = {}
for line in structured.strip().split("\n"):
    if ":" in line:
        key, _, value = line.partition(":")
        result[key.strip()] = value.strip()

print("Parsed:")
for k, v in result.items():
    print(f"  {k}: {v}")


---
## Summary

| Tip | Bad | Good |
|---|---|---|
| **1. Be specific** | `"Explain NLP"` | `"Explain NLP in 2 sentences for a beginner"` |
| **2. System prompt** | Repeat instructions every message | Set once in `system=` |
| **3. Give a role** | `"You are an assistant"` | `"You are a biomedical NLP expert"` |
| **4. Use delimiters** | Mix instruction and data | Wrap data in `<text>...</text>` |
| **5. Specify format** | Hope for JSON | `"Output only valid JSON"` |

Next: **03_prompting_examples.ipynb** — zero-shot, few-shot, and chain-of-thought.
